In [1]:
import sys
from pathlib import Path

# Add src to path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT / 'src'))

from pipeline import Retriever, load_chunks

# 1. Load chunks and build the index
# This will take a few minutes the very first time because it downloads the models.
R = Retriever(load_chunks())

# 2. Test Dense Search
questions = [
    'what are the employer duties regarding protective equipment?',
    'who appoints safety representatives?'
]

for q in questions:
    print(f"\nQuestion: {q}")
    for cid in R.search(q, 'dense', 5):
        print(' ', R.by_id[cid]['citation'], '|', R.by_id[cid]['section_title'])

c:\Users\Kagimu Brian\anaconda3\envs\osh_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2892.48it/s]


Loading re-ranker model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3984.20it/s]


Loaded existing ChromaDB index.

Question: what are the employer duties regarding protective equipment?
  OSH Act 2006, Part III, s.19 | Employer to provide protective gear
  OSH Act 2006, Part XII, s.91 | Provision of personal protective gear
  OSH Act 2006, Part III, s.13 | Duty of employers to protect workers
  OSH Act 2006, Part III, s.18 | Employer to monitor and control the release of dangerous substances into the
  OSH Act 2006, Part IV, s.23 | Employers’ duty to persons other than employees

Question: who appoints safety representatives?
  OSH Act 2006, Part III, s.15 | Safety representatives
  OSH Act 2006, Part III, s.16 | Safety committees
  OSH Act 2006, Part II, s.3 | Appointment of inspectors
  OSH Act 2006, Part II, s.10 | Establishment of the Occupational Safety and Health Board
  OSH Act 2006, Part II, s.11 | Advisory panels


In [2]:
q = 'What are the employer duties regarding protective equipment?'

print(f"--- Comparing modes for: '{q}' ---\n")

for mode in ('dense', 'bm25', 'hybrid', 'hybrid_rerank'):
    print(f"Mode: {mode}")
    for cid in R.search(q, mode, 5):
        print(f"  {R.by_id[cid]['citation']} | {R.by_id[cid]['section_title']}")
    print()

--- Comparing modes for: 'What are the employer duties regarding protective equipment?' ---

Mode: dense
  OSH Act 2006, Part III, s.19 | Employer to provide protective gear
  OSH Act 2006, Part XII, s.91 | Provision of personal protective gear
  OSH Act 2006, Part III, s.13 | Duty of employers to protect workers
  OSH Act 2006, Part III, s.18 | Employer to monitor and control the release of dangerous substances into the
  OSH Act 2006, Part IV, s.23 | Employers’ duty to persons other than employees

Mode: bm25
  OSH Act 2006, Part III, s.19 | Employer to provide protective gear
  OSH Act 2006, Part XII, s.91 | Provision of personal protective gear
  OSH Act 2006, Part III, s.13 | Duty of employers to protect workers
  OSH Act 2006, Part II, s.8 | Causing delay to, or obstruction of inspectors
  OSH Act 2006, Part XIII, s.96 | Provision of chemical data sheets

Mode: hybrid
  OSH Act 2006, Part III, s.19 | Employer to provide protective gear
  OSH Act 2006, Part XII, s.91 | Provision o

In [3]:
# 5 In-scope questions (should have HIGH scores)
in_scope = [
    'what are the employer duties regarding protective equipment?',
    'who appoints safety representatives?',
    'what are the rules for fencing machinery?',
    'how should accidents be reported?',
    'what is the penalty for obstructing an inspector?'
]

# 5 Out-of-scope questions (should have LOW scores)
out_scope = [
    'How do I register a company?',
    'What is the income tax rate in Uganda?',
    'How do I get a passport?',
    'What is the speed limit on the highway?',
    'Who won the football match yesterday?'
]

print("--- TOP RERANKER SCORES ---")
for label, qs in (('IN ', in_scope), ('OUT', out_scope)):
    for q in qs:
        ids, scores = R.hybrid(q)
        # Print the highest score for each question
        print(f"{label} | Score: {scores[0]:.2f} | {q}")

--- TOP RERANKER SCORES ---
IN  | Score: 7.18 | what are the employer duties regarding protective equipment?
IN  | Score: 5.29 | who appoints safety representatives?
IN  | Score: 4.76 | what are the rules for fencing machinery?
IN  | Score: -5.19 | how should accidents be reported?
IN  | Score: 4.19 | what is the penalty for obstructing an inspector?
OUT | Score: -2.22 | How do I register a company?
OUT | Score: -11.15 | What is the income tax rate in Uganda?
OUT | Score: -10.58 | How do I get a passport?
OUT | Score: -10.85 | What is the speed limit on the highway?
OUT | Score: -10.90 | Who won the football match yesterday?


In [4]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.append(str(ROOT / 'src'))

# Import the new answer function
from pipeline import answer, get_retriever
R = get_retriever()

test_questions = [
    'what are the employer duties regarding protective equipment?', # In-scope
    'How do I register a company?',                                # Out-of-scope (Should refuse)
    'who appoints safety representatives?'                         # In-scope
]

for q in test_questions:
    print(f"\n========================================\nQ: {q}")
    o = answer(q, R)
    print(f"Refused: {o['abstained']} | Latency: {o['latency_s']}s | Cost: ${o['cost_usd']:.5f}")
    print(f"A: {o['answer']}")
    print(f"Citations: {[R.by_id[i]['citation'] for i in o['ids']]}")

Loaded existing ChromaDB index.

Q: what are the employer duties regarding protective equipment?
Refused: False | Latency: 4.72s | Cost: $0.00025
A: Employers have several duties regarding protective equipment:

1. They must provide adequate and suitable protective clothing and equipment when air pollution and chemical substances exceed specified limits [1].
2. Employers are responsible for ensuring that personal protective equipment is used whenever required [1].
3. They must supply and maintain personal protective equipment free of charge to employees and provide instructions for its use [3].
4. Employers should also ensure that the working environment is safe and free from risks, which includes providing appropriate protective equipment [2].

The passages do not specify the types of protective equipment required or the specific circumstances under which they must be provided.
Citations: ['OSH Act 2006, Part III, s.19', 'OSH Act 2006, Part III, s.13', 'OSH Act 2006, Part XII, s.91', 